# **1. Load the Dataset**

In [65]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/SEM6_AI/Week8/trum_tweet_sentiment_analysis.csv")

In [66]:
df.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


**Deleting rows that contain empty values**

In [67]:
data_cleaning = df["text"].dropna()

In [68]:
data_cleaning[0]

'RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrump\x85 https://t.co/gFBvUkMX9z'

# **2. Text Cleaning and tokenization**

**Lower Order**

In [69]:
def lower_order(text):
  """
  This function converts all the text in input text to lower order.
  Input Args:
  token_text : input text.
  Returns:
  small_order_text : text converted to small/lower order.
  """
  small_order_text = text.lower()
  return small_order_text

# Test:
sample_text = "This Is some Normalized TEXT"
sample_small = lower_order(sample_text)
print(sample_small)


this is some normalized text


**Remove URLs**

In [70]:
import re
def remove_urls(text):
  """
  This function will try to remove URL present in out dataset and replace it with space using regex library.
  Input Args:
  text: strings of text that may contain URLs.
  Output Args:
  text: URLs replaces with text
  """
  url_pattern = re.compile(r'https?://\S+|www\.\S+')
  return url_pattern.sub(r'', text)


**Remove Emojis**

In [71]:
def remove_emoji(string):
  """
  This function will replace the emoji in string with whitespace
  """
  emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)

**Remove Unwanted Characters**

In [72]:
def removeunwanted_characters(document):
  """
  This function will remove all the unwanted characters from the input dataset.
  Input Args:
  documet: A text data to be cleaned.
  Return:
  A cleaned document.
  """
  # remove user mentions
  document = re.sub("@[A-Za-z0-9_]+"," ", document)
  # remove hashtags
  document = re.sub("#[A-Za-z0-9_]+","", document)
  # remove punctuation
  document = re.sub("[^0-9A-Za-z ]", "" , document)
  #remove emojis
  document = remove_emoji(document)
  # remove double spaces
  document = document.replace('  ',"")
  return document.strip()       # Removes extra spaces from the start and end of a string.

**Tokenization**

In [73]:
import nltk #the main NLP library (toolbox)
nltk.download('punkt_tab')  #punkt provides the rules/data needed to split text properly
from nltk import word_tokenize #worker (function that uses the rules of punkt)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Lemmatization**

In [74]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

def lemmatization(token_text):
  """
  This function performs the lemmatization operations as explained above.
  Input Args:
  token_text: list of tokens.
  Returns:
  lemmatized_tokens: list of lemmatized tokens.
  """
  lemma_tokens = []
  wordnet = WordNetLemmatizer()
  lemmatized_tokens = [wordnet.lemmatize(token, pos = 'v') for token in token_text]

  return lemmatized_tokens

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


**Stemming**

In [75]:
from nltk.stem import PorterStemmer

def stemming(text):
  """
  This function performs stemming operations.
  Input Args:
  token_text: list of tokenize text.
  Returns:
  stemm_tokes: list of stemmed tokens.
  """
  porter = PorterStemmer()
  stemm_tokens = []
  for word in text:
    stemm_tokens.append(porter.stem(word))
  return stemm_tokens

**Remove Stopwords**

In [76]:
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))
custom_stopwords = ['@', 'RT']
stop_words.update(custom_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [77]:
def remove_stopwords(text_tokens):
  """
  This function removes all the stopwords present in out text tokens.
  Input Args:
  text_tokens: tokenize input of our datasets.
  Returns:
  result_tokens: list of token without stopword.
  """

  result_tokens = []
  for token in text_tokens:
    if token not in stop_words:
       result_tokens.append(token)
  return result_tokens

In [78]:
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer

def text_cleaning_pipeline(text, rule="lemmatize"):

    wordnet = WordNetLemmatizer()
    porter = PorterStemmer()

    text = lower_order(text)
    text = remove_urls(text)
    text = remove_emoji(text)
    text = removeunwanted_characters(text)

    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]

    if rule == "lemmatize":
        tokens = [wordnet.lemmatize(w, pos='v') for w in tokens]
    elif rule == "stem":
        tokens = [porter.stem(w) for w in tokens]

    return " ".join(tokens)

In [79]:
df["cleaned_text"] = df["text"].apply(text_cleaning_pipeline)

In [80]:
df.head()

,text,Sentiment,cleaned_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,0,rtnot drain swamp taxpayer dollars trip advert...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0,icymi hackers rig fm radio station play antitr...
2,Trump protests: LGBTQ rally in New York https:...,1,trump protest lgbtq rally new yorkbyvia
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0,hi im piers morgan david beckham awful donald ...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0,rt tech firm sue buzzfeed publish unverified t...


# **3. Train-Test Split**

In [81]:
from sklearn.model_selection import train_test_split

X = df["cleaned_text"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# **4. TF-IDF Vectorization**

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# **5. Model Training and Evaluation**

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

#Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

#Predict
y_pred = model.predict(X_test_vec)

#Evaluate
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.97      0.96    248563
           1       0.93      0.90      0.92    121462

    accuracy                           0.95    370025
   macro avg       0.94      0.94      0.94    370025
weighted avg       0.95      0.95      0.95    370025



# **Importance of Text pre-processing in NLP**

In [87]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [88]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [86]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

#Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

#Predict
y_pred = model.predict(X_test_vec)

#Evaluate
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96    248563
           1       0.93      0.92      0.93    121462

    accuracy                           0.95    370025
   macro avg       0.95      0.94      0.95    370025
weighted avg       0.95      0.95      0.95    370025



Text preprocessing plays a crucial role in improving the quality and consistency of input data in Natural Language Processing tasks. It involves steps such as lowercasing, removal of URLs, emojis, punctuation, stopwords, and applying techniques like stemming or lemmatization.


In this experiment, the model was trained both with and without preprocessing. While the overall accuracy remained similar (0.95 in both cases), preprocessing significantly improves text standardization by reducing noise and vocabulary redundancy. This ensures that semantically similar words are treated consistently (e.g., “running”, “runs”, “ran” → “run”), which helps machine learning models learn more meaningful patterns.


Although raw text may sometimes retain strong emotional cues (such as emojis or punctuation), it also introduces inconsistency and noise, which can affect model generalization in real-world scenarios. Therefore, preprocessing is essential for building robust and reliable NLP systems.